# Train the MLBB vision detector (free, in Google Colab)

Makes `mlbb_detector.tflite` + `labels.txt` for the MLBB Research app (Model page → *Choose detector file*).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free).
2. Make a free account at roboflow.com and copy your **API key** (Settings → API Keys). You type it into the box below; it is never saved in this notebook.
3. Default dataset: **Mobile Legends Minimap Dataset** by *gladi* on Roboflow Universe, licence **CC BY 4.0** (credit the author if you share the model).

Run each cell from top to bottom (tap ▶). Training takes about 10–20 minutes.

In [ ]:
!pip install -q ultralytics roboflow

## 1. Settings
Change these only if you want a different Roboflow dataset. For a full-screen dataset, also set the app's *Detector looks at* to *the whole screen*.

In [ ]:
WORKSPACE = "gladi"                            # Roboflow workspace (from the dataset URL)
PROJECT   = "mobile-legends-minimap-dataset"   # Roboflow project (from the dataset URL)
VERSION   = 2                                  # dataset version number
EPOCHS    = 80                                 # more = better but slower
IMGSZ     = 320                                # 320 is fast on phones; 640 is more accurate

## 2. Download the dataset

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Paste your Roboflow API key and press Enter: "))
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
print("Downloaded to", dataset.location)

## 2b. Add YOUR OWN pictures (optional, recommended)
The app saves minimap pictures with boxes already drawn while you play (Learn page → *Collect detector training pictures*), in `Download/MLBB-Research/dataset` (`images/`, `labels/`, `classes.txt`).

**Best:** upload that folder to your Roboflow project first, fix the wrong / missing boxes there, and export a new version (then just change `VERSION` above).

**Quick:** zip the `dataset` folder on your phone (file manager → compress), run the next cell and choose the zip. Its pictures are added to the training set as they are.

In [ ]:
import zipfile, glob, os, random, shutil, yaml
from google.colab import files
up = files.upload()                      # choose the zip of Download/MLBB-Research/dataset
zname = next(k for k in up if k.endswith(".zip"))
shutil.rmtree("/content/mine", ignore_errors=True)
zipfile.ZipFile(zname).extractall("/content/mine")
imgs = sorted(glob.glob("/content/mine/**/images/*.png", recursive=True))
print(len(imgs), "of your pictures")

# your class names must match the Roboflow dataset's
rf_yaml = yaml.safe_load(open(f"{dataset.location}/data.yaml"))
names = rf_yaml["names"]
cls_files = glob.glob("/content/mine/**/classes.txt", recursive=True)
mine_names = [l.strip() for l in open(cls_files[0]) if l.strip()] if cls_files else list(names)
# Your boxes are numbered by the app's class list; the dataset may have other classes / another
# order. Convert each box by its NAME; boxes of a class the dataset does not have are dropped.
remap = {i: list(names).index(n) for i, n in enumerate(mine_names) if n in list(names)}
if mine_names != list(names):
    print("Your classes:", mine_names, "-> dataset classes:", list(names))
    print("Converted by name:", {mine_names[i]: names[j] for i, j in remap.items()},
          "| dropped:", [n for n in mine_names if n not in list(names)])

# 85% of your pictures go to training, 15% to checking
random.seed(0); random.shuffle(imgs)
for part, lst in (("train", imgs[:int(len(imgs) * 0.85)]), ("valid", imgs[int(len(imgs) * 0.85):])):
    os.makedirs(f"/content/mine_{part}/images", exist_ok=True); os.makedirs(f"/content/mine_{part}/labels", exist_ok=True)
    for im in lst:
        lb = im.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        if not os.path.exists(lb): continue
        rows = []
        for line in open(lb):
            v = line.split()
            if len(v) == 5 and int(v[0]) in remap: rows.append(" ".join([str(remap[int(v[0])])] + v[1:]))
        shutil.copy(im, f"/content/mine_{part}/images/")
        open(f"/content/mine_{part}/labels/" + os.path.basename(lb), "w").write("\n".join(rows) + ("\n" if rows else ""))

def rf_dir(k):
    v = rf_yaml[k]
    return v if os.path.isabs(v) else os.path.normpath(os.path.join(dataset.location, v.replace("../", "")))
combined = {"train": [rf_dir("train"), "/content/mine_train/images"],
            "val": [rf_dir("val"), "/content/mine_valid/images"],
            "nc": len(names), "names": names}
yaml.safe_dump(combined, open("/content/combined.yaml", "w"))
DATA = "/content/combined.yaml"
print("Training on the Roboflow set + your pictures:", DATA)

## 3. Train a small YOLOv8 model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # 'nano' = smallest and fastest, good for phones
DATA = globals().get("DATA", f"{dataset.location}/data.yaml")   # your pictures too, if added above
model.train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=20)
best_path = model.trainer.best
print("Best weights:", best_path)

## 4. Export to LiteRT (.tflite) and download

In [ ]:
import glob, os, shutil, yaml

best = YOLO(best_path)
# Newer Ultralytics calls this format "litert"; older versions call it "tflite"
try:
    out = best.export(format="litert", imgsz=IMGSZ)
except Exception as e:
    print("litert export not available here, using tflite:", e)
    out = best.export(format="tflite", imgsz=IMGSZ)
print("Export result:", out)

# find the .tflite wherever Ultralytics saved it
weights_dir = os.path.dirname(str(best_path))
found = []
if out and str(out).endswith(".tflite"):
    found.append(str(out))
found += sorted(glob.glob(weights_dir + "/**/*.tflite", recursive=True))
found += sorted(glob.glob("/content/**/*.tflite", recursive=True))
found = [f for f in found if os.path.exists(f) and not f.endswith("mlbb_detector.tflite")]
if not found:
    print("No .tflite found. Files next to best.pt:")
    for root, dirs, fs in os.walk(weights_dir):
        for f in fs:
            print(os.path.join(root, f))
    raise SystemExit("Send this list to Claude.")
float32 = [f for f in found if "float32" in f]
pick = (float32 or found)[0]
shutil.copy(pick, "/content/mlbb_detector.tflite")
print("Using:", pick, round(os.path.getsize(pick) / 1e6, 1), "MB")

names = yaml.safe_load(open(f"{dataset.location}/data.yaml"))["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
open("/content/labels.txt", "w").write(",".join(names))
print("Classes (paste into the app if they differ from the default):")
print(",".join(names))

## 5. Download the files
On a phone the direct download often does nothing. If so, run the **Google Drive** cell instead: the files then appear in your Google Drive (My Drive), and the Drive app can save them to the phone.

In [ ]:
# A. Direct download (works best on a computer)
from google.colab import files
files.download("/content/mlbb_detector.tflite")
files.download("/content/labels.txt")

In [ ]:
# B. Save to Google Drive (most reliable on a phone; asks you to allow Drive access)
from google.colab import drive
drive.mount("/content/drive")
shutil.copy("/content/mlbb_detector.tflite", "/content/drive/MyDrive/mlbb_detector.tflite")
shutil.copy("/content/labels.txt", "/content/drive/MyDrive/mlbb_labels.txt")
print("Saved to Google Drive > My Drive: mlbb_detector.tflite and mlbb_labels.txt")

## 6. Put it in the app
1. Copy `mlbb_detector.tflite` to your phone (it downloads straight to the phone if you run Colab there).
2. App → **Model** page → *Choose detector file (.tflite)*.
3. If the class list printed above differs from the app's default, paste it into *Class names*.
4. Home page preview shows the detections as cyan boxes.